# Phase 0 — Exploratory Data Analysis & Dataset Audit

**Goal:** before a single table is built, *prove from the raw files* every
data decision the rest of the pipeline depends on. Nothing here is taken on
faith — each claim below is recomputed live from `raw_data/` so the audit is
fully reproducible.

This notebook is the evidence behind the "Phase 0 — locked decisions" that
`wyscout_lib.py` implements. The seven decisions it establishes:

| # | Decision | Proven in |
|---|---|---|
| 1 | **Goals come from the match-sheet lineup, not event tags** (tag 101 also fires on conceded shots & shootouts) | §6 |
| 2 | **Shot-on-target = tag 1801** (every shot carries exactly one of 1801/1802) | §8 |
| 3 | **Tag 1801 on a duel ≠ "duel won"** (both players can carry it) → DEFENDING uses volume, not win-rate | §7 |
| 4 | **Assists come from event tag 301**, because the lineup `assists` field is empty in the leagues | §4 |
| 5 | **Cards store the *minute*, not a count**; red-card minute trims minutes played | §4 |
| 6 | **Coordinates are team-relative** (attack always toward x=100) → universal spatial flags | §9 |
| 7 | **Periods are `E1`/`E2`/`P`** (extra time → cap minutes at 120, shootout goals excluded) | §5, §10 |

Plus two cleaning facts: **names are double-escaped unicode** (§2) and
**237 team-match entries have no coachId** (§4).

In [1]:
import sys, json, zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

sys.path.insert(0, str(Path.cwd()))
import wyscout_lib as wl

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

RAW = wl.RAW
print("raw_data at:", RAW)
print("files:", sorted(p.name for p in RAW.iterdir()))

raw_data at: /Users/preynsh.thukral/Desktop/StatMagic/wyscout-soccer-match-event-dataset/raw_data
files: ['README.md', 'events.zip', 'matches.zip', 'players.json', 'teams.json']


## 1. File inventory

Four raw inputs: two flat JSON files (players, teams) and two zips (matches,
events) each holding one JSON per competition. We confirm the shape of every
file before trusting any of it.

In [2]:
players = json.load(open(RAW / "players.json", encoding="utf-8"))
teams   = json.load(open(RAW / "teams.json", encoding="utf-8"))
print(f"players.json : {len(players):,} players")
print(f"teams.json   : {len(teams):,} teams")

inv = []
ENR = wl.DATA / "events_enriched"          # row-identical to the raw event files
with zipfile.ZipFile(RAW / "matches.zip") as mz:
    for cid, meta in wl.COMPETITIONS.items():
        stem = meta["file"]
        n_matches = len(json.load(mz.open(f"matches_{stem}.json")))
        # event counts read from the cached enriched parquet metadata (instant,
        # and 1:1 with raw events) — avoids re-parsing 3.25M raw events here.
        enr = ENR / f"{cid}.parquet"
        n_events = pq.ParquetFile(enr).metadata.num_rows if enr.exists() else None
        inv.append({"competition": cid, "name": meta["name"], "season": meta["season"],
                    "type": meta["type"], "matches": n_matches, "events": n_events})
inv = pd.DataFrame(inv)
print(f"\nTOTAL: {inv.matches.sum():,} matches · "
      f"{int(inv.events.sum()):,} events" if inv.events.notna().all() else "")
inv

players.json : 3,603 players
teams.json   : 142 teams

TOTAL: 1,941 matches · 3,251,294 events


,competition,name,season,type,matches,events
0,england,English Premier League,2017/18,league,380,643150
1,france,French Ligue 1,2017/18,league,380,632807
2,germany,German Bundesliga,2017/18,league,306,519407
3,italy,Italian Serie A,2017/18,league,380,647372
4,spain,Spanish La Liga,2017/18,league,380,628659
5,euro_2016,UEFA Euro 2016,2016,tournament,51,78140
6,world_cup,FIFA World Cup 2018,2018,tournament,64,101759


## 2. `players.json` — structure & the unicode-escape repair

One object per player. `role.code2` is the only position granularity Wyscout
gives us: **GK / DF / MD / FW** — this single field drives every
position-normalised percentile in the ratings.

In [3]:
print("=== one raw player ===")
print(json.dumps(players[0], indent=2, ensure_ascii=False))

pos = Counter((p.get("role") or {}).get("code2") for p in players)
print("\nposition_code distribution:", dict(pos))

=== one raw player ===
{
  "passportArea": {
    "name": "Turkey",
    "id": "792",
    "alpha3code": "TUR",
    "alpha2code": "TR"
  },
  "weight": 78,
  "firstName": "Harun",
  "middleName": "",
  "lastName": "Tekin",
  "currentTeamId": 4502,
  "birthDate": "1989-06-17",
  "height": 187,
  "role": {
    "code2": "GK",
    "code3": "GKP",
    "name": "Goalkeeper"
  },
  "birthArea": {
    "name": "Turkey",
    "id": "792",
    "alpha3code": "TUR",
    "alpha2code": "TR"
  },
  "wyId": 32777,
  "foot": "right",
  "shortName": "H. Tekin",
  "currentNationalTeamId": 4687
}

position_code distribution: {'GK': 426, 'DF': 1200, 'MD': 1257, 'FW': 720}


**The encoding trap.** The source JSON *double-escapes* accents: a name like
"Aréola" is stored as the literal 6-character sequence `Aréola` (a real
backslash, then `u00e9`), not as the character `é`. Left unrepaired, cards
render garbage. We measure exactly how many players are affected and confirm
the regex repair (`wl._decode_name`) fixes it.

In [4]:
name_fields = ("firstName", "lastName", "shortName")
affected = [p for p in players if any("\\u" in str(p.get(f, "")) for f in name_fields)]
print(f"players with a \\uXXXX escape in any name field: {len(affected):,} / {len(players):,}")
for f in name_fields:
    n = sum("\\u" in str(p.get(f, "")) for p in players)
    print(f"  {f:10s}: {n:,}")

ex = next(p for p in affected)
print("\nexample repair:")
print(f"  raw shortName : {ex['shortName']!r}")
print(f"  decoded       : {wl._decode_name(ex['shortName'])!r}")
# nationalities are affected too (e.g. Côte d'Ivoire)
nat_aff = {(p.get("passportArea") or {}).get("name") for p in players
           if "\\u" in str((p.get("passportArea") or {}).get("name", ""))}
print(f"\naffected nationalities (sample): "
      f"{[wl._decode_name(n)+' <- '+n for n in list(nat_aff)[:3]]}")

players with a \uXXXX escape in any name field: 1,152 / 3,603
  firstName : 531
  lastName  : 828
  shortName : 722

example repair:
  raw shortName : 'I. Konat\\u00e9'
  decoded       : 'I. Konaté'

affected nationalities (sample): ['Curaçao <- Cura\\u00e7ao', 'São Tomé e Príncipe <- S\\u00e3o Tom\\u00e9 e Pr\\u00edncipe', "Côte d'Ivoire <- C\\u00f4te d'Ivoire"]


## 3. `teams.json` — structure

Clubs and national teams in one file. We only need `wyId` → `name` downstream
(event rows reference teams by id; coach cards show the team name).

In [5]:
print("=== one raw team ===")
print(json.dumps(teams[0], indent=2, ensure_ascii=False))
print("\nteam type distribution:", dict(Counter(t.get("type") for t in teams)))

=== one raw team ===
{
  "city": "Newcastle upon Tyne",
  "name": "Newcastle United",
  "wyId": 1613,
  "officialName": "Newcastle United FC",
  "area": {
    "name": "England",
    "id": "0",
    "alpha3code": "XEN",
    "alpha2code": ""
  },
  "type": "club"
}

team type distribution: {'club': 98, 'national': 44}


## 4. `matches.zip` — the match sheet & its encoding traps

The richest file. Each match nests `teamsData[teamId]` with the side, the
final/ET/penalty scores, the `coachId`, and a `formation` block holding the
**lineup**, **bench**, and **substitutions**. The per-player lineup entries
are where goals and cards live — but two of those fields are *not what they
look like*.

In [6]:
with zipfile.ZipFile(RAW / "matches.zip") as mz:
    wc_matches = json.load(mz.open("matches_World_Cup.json"))
m = next(x for x in wc_matches if x["label"].startswith("France - Croatia"))
print("top-level keys:", list(m.keys()))
side = next(td for td in m["teamsData"].values() if td["side"] == "home")
print("\n=== one teamsData entry (home) — formation trimmed ===")
print(json.dumps({k: (v if k != "formation" else "<lineup/bench/substitutions>")
                  for k, v in side.items()}, indent=2, ensure_ascii=False))
print("\n=== one lineup player entry ===")
print(json.dumps(side["formation"]["lineup"][0], indent=2))
print("\n=== one substitution entry ===")
print(json.dumps(side["formation"]["substitutions"][0], indent=2))

top-level keys: ['status', 'roundId', 'gameweek', 'teamsData', 'seasonId', 'dateutc', 'winner', 'venue', 'wyId', 'label', 'date', 'groupName', 'referees', 'duration', 'competitionId']

=== one teamsData entry (home) — formation trimmed ===
{
  "scoreET": 0,
  "coachId": 25549,
  "side": "home",
  "teamId": 4418,
  "score": 4,
  "scoreP": 0,
  "hasFormation": 1,
  "formation": "<lineup/bench/substitutions>",
  "scoreHT": 2
}

=== one lineup player entry ===
{
  "playerId": 31528,
  "assists": "0",
  "goals": "0",
  "ownGoals": "0",
  "redCards": "0",
  "yellowCards": "27"
}

=== one substitution entry ===
{
  "playerIn": 8200,
  "assists": "0",
  "playerOut": 31528,
  "minute": 55
}


**Trap A — `goals` is a real count, but `yellowCards`/`redCards` are MINUTES.**
If we treated cards as counts, every booked player would look like they'd seen
dozens of cards. The value distribution proves they're match-minutes (0 = none,
otherwise 1–120).

In [7]:
gvals, yvals, rvals = Counter(), Counter(), Counter()
for mt in wc_matches:
    for td in mt["teamsData"].values():
        for p in td["formation"]["lineup"] + td["formation"]["bench"]:
            gvals[str(p.get("goals"))]   += 1
            yvals[str(p.get("yellowCards"))] += 1
            rvals[str(p.get("redCards"))] += 1
print("goals field values        :", dict(sorted(gvals.items(), key=lambda x: str(x[0]))))
print("  -> small integers => a genuine COUNT")
nz_yellow = sorted(int(k) for k in yvals if k not in ("0", "None") and int(k) > 0)
print(f"\nyellowCards non-zero values: {nz_yellow[:15]}{' …' if len(nz_yellow)>15 else ''}")
print(f"  range {min(nz_yellow)}–{max(nz_yellow)}  => these are MINUTES, not counts")
print(f"  (so we read cards as 0/1 occurrence, and a red-card minute trims minutes played)")

goals field values        : {'0': 798, '1': 131, '2': 10, '3': 2, 'null': 1944}
  -> small integers => a genuine COUNT

yellowCards non-zero values: [1, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 21, 22, 23, 25] …
  range 1–118  => these are MINUTES, not counts
  (so we read cards as 0/1 occurrence, and a red-card minute trims minutes played)


**Trap B — the lineup `assists` field is empty in the leagues.** It's only
populated for the two tournaments, so a uniform assist signal has to come from
the events (tag 301, see §6). We sum the lineup `assists` per competition to
show the hole.

In [8]:
rows = []
with zipfile.ZipFile(RAW / "matches.zip") as mz:
    for cid, meta in wl.COMPETITIONS.items():
        data = json.load(mz.open(f"matches_{meta['file']}.json"))
        tot = 0
        for mt in data:
            for td in mt["teamsData"].values():
                for p in td["formation"]["lineup"] + td["formation"]["bench"]:
                    tot += wl._as_int(p.get("assists"))
        rows.append({"competition": cid, "type": meta["type"], "lineup_assists_sum": tot})
asum = pd.DataFrame(rows)
print(asum.to_string(index=False))
print("\n=> leagues report 0 lineup assists; only tournaments populate it.")
print("   Conclusion: derive assists from event tag 301 (uniform across all 7).")

competition       type  lineup_assists_sum
    england     league                   0
     france     league                   0
    germany     league                   0
      italy     league                   0
      spain     league                   0
  euro_2016 tournament                  79
  world_cup tournament                  81

=> leagues report 0 lineup assists; only tournaments populate it.
   Conclusion: derive assists from event tag 301 (uniform across all 7).


**Trap C — 237 team-match entries have no `coachId`.** These are excluded from
coach ratings (a coach can't be credited for a match we can't attribute).

In [9]:
miss = tot = 0
with zipfile.ZipFile(RAW / "matches.zip") as mz:
    for cid, meta in wl.COMPETITIONS.items():
        for mt in json.load(mz.open(f"matches_{meta['file']}.json")):
            for td in mt["teamsData"].values():
                tot += 1
                if not td.get("coachId"):
                    miss += 1
print(f"team-match entries: {tot:,}  |  missing coachId: {miss}  ({100*miss/tot:.1f}%)")

team-match entries: 3,882  |  missing coachId: 237  (6.1%)


## 5. `events.zip` — the event catalogue

One object per on-ball action: an `eventName`/`subEventName`, a timestamp, 1–2
`positions` (start/end), and a list of integer `tags` encoding outcome, zone,
and body part. We catalogue every event type, every tag, and every period in
the World Cup file.

In [10]:
with zipfile.ZipFile(RAW / "events.zip") as ez:
    wc_events = json.load(ez.open("events_World_Cup.json"))
print(f"World Cup events: {len(wc_events):,}\n")
print("=== one raw event ===")
print(json.dumps(wc_events[0], indent=2))

World Cup events: 101,759

=== one raw event ===
{
  "eventId": 8,
  "subEventName": "Simple pass",
  "tags": [
    {
      "id": 1801
    }
  ],
  "playerId": 122671,
  "positions": [
    {
      "y": 50,
      "x": 50
    },
    {
      "y": 53,
      "x": 35
    }
  ],
  "matchId": 2057954,
  "eventName": "Pass",
  "teamId": 16521,
  "matchPeriod": "1H",
  "eventSec": 1.6562140000000003,
  "subEventId": 85,
  "id": 258612104
}


In [11]:
types = Counter((e["eventName"], e.get("subEventName", "")) for e in wc_events)
print("event (name, subName) catalogue:")
for (n, s), c in sorted(types.items()):
    print(f"  {n:24s} {s:24s} {c:>7,}")

tags = Counter(t["id"] for e in wc_events for t in e.get("tags", []))
print("\nall tag ids present:", sorted(tags))

periods = Counter(e["matchPeriod"] for e in wc_events)
print("\nperiods present:", dict(periods))
print("  NOTE: extra time is coded E1/E2 (NOT ET1/ET2) and the shootout is P.")
print("  This is why the code constants are EXTRA_TIME_PERIODS={'E1','E2'}, SHOOTOUT='P'.")

event (name, subName) catalogue:
  Duel                     Air duel                   5,518
  Duel                     Ground attacking duel      8,072
  Duel                     Ground defending duel      8,007
  Duel                     Ground loose ball duel     4,330
  Foul                     Foul                       1,634
  Foul                     Hand foul                     79
  Foul                     Late card foul                 7
  Foul                     Out of game foul              17
  Foul                     Protest                       15
  Foul                     Simulation                     1
  Foul                     Time lost foul                10
  Foul                     Violent Foul                   3
  Free Kick                Corner                       603
  Free Kick                Free Kick                  1,199
  Free Kick                Free kick cross              292
  Free Kick                Free kick shot                86
  Free 

## 6. The goal-counting investigation  (decision #1 — the headline)

Goal tag **101** is the obvious way to count goals — and it's wrong. It also
fires on the **goalkeeper's Save attempt** for every conceded goal, and on
**penalty-shootout** kicks. Counting raw tag-101 events roughly *doubles* the
real total. We show the leak, then validate the two correct sources.

In [12]:
tag101 = Counter((e["eventName"], e["matchPeriod"])
                 for e in wc_events if any(t["id"] == 101 for t in e["tags"]))
print("tag-101 events by (eventName, period):")
for k, c in sorted(tag101.items(), key=lambda x: -x[1]):
    print(f"  {k[0]:16s} {k[1]:3s} {c:>4}")
naive = sum(tag101.values())
print(f"\nnaïve count of ALL tag-101 events: {naive}")
print("  -> includes Save attempt (conceded) and shootout (P): NOT competitive goals scored.")

tag-101 events by (eventName, period):
  Save attempt     2H   101
  Shot             2H    82
  Save attempt     1H    65
  Shot             1H    43
  Free Kick        P     26
  Save attempt     P     26
  Free Kick        1H    15
  Free Kick        2H    14
  Shot             E2     2
  Save attempt     E2     2
  Shot             E1     1
  Save attempt     E1     1

naïve count of ALL tag-101 events: 378
  -> includes Save attempt (conceded) and shootout (P): NOT competitive goals scored.


**The two correct sources, cross-validated.**
- *Event-derived* goal = tag 101 on a **Shot or Free Kick**, excluding period **P**.
- *Authoritative* goal = the lineup `goals` count from the match sheet.

They must agree, and they do.

In [13]:
event_goals = sum(1 for e in wc_events
                  if any(t["id"] == 101 for t in e["tags"])
                  and e["eventName"] in ("Shot", "Free Kick")
                  and e["matchPeriod"] != "P")
lineup_goals = sum(wl._as_int(p.get("goals"))
                   for mt in wc_matches
                   for td in mt["teamsData"].values()
                   for p in td["formation"]["lineup"] + td["formation"]["bench"])
own_goals = sum(wl._as_int(p.get("ownGoals"))
                for mt in wc_matches
                for td in mt["teamsData"].values()
                for p in td["formation"]["lineup"] + td["formation"]["bench"])
print(f"event-derived goals (Shot/FK, not P): {event_goals}")
print(f"authoritative lineup goals          : {lineup_goals}")
print(f"lineup own goals (excluded)         : {own_goals}")
print(f"official WC 2018 total (goals+OG)   : {lineup_goals + own_goals}  "
      f"(the real tournament total is 169)")
print(f"\nevent vs authoritative agreement    : "
      f"{100*min(event_goals,lineup_goals)/max(event_goals,lineup_goals):.1f}%")

event-derived goals (Shot/FK, not P): 157
authoritative lineup goals          : 157
lineup own goals (excluded)         : 12
official WC 2018 total (goals+OG)   : 169  (the real tournament total is 169)

event vs authoritative agreement    : 100.0%


**Named ground truth — Harry Kane scored 6 at the 2018 World Cup.** Both the
authoritative source and the filtered-event source reproduce exactly 6.

In [14]:
kane_id = next(p["wyId"] for p in players if p.get("shortName") == "H. Kane")
kane_lineup = sum(wl._as_int(p.get("goals"))
                  for mt in wc_matches
                  for td in mt["teamsData"].values()
                  for p in td["formation"]["lineup"] + td["formation"]["bench"]
                  if p["playerId"] == kane_id)
kane_event = sum(1 for e in wc_events
                 if e["playerId"] == kane_id and any(t["id"] == 101 for t in e["tags"])
                 and e["eventName"] in ("Shot", "Free Kick") and e["matchPeriod"] != "P")
print(f"Kane (id {kane_id}) WC goals — lineup: {kane_lineup} · event-derived: {kane_event}  (expected 6)")

Kane (id 8717) WC goals — lineup: 6 · event-derived: 6  (expected 6)


## 7. Tag 1801 is *accuracy*, not "duel won"  (decision #3)

It's tempting to read tag 1801 on a duel as "won the duel." But a duel is a
contest between two players and **both** event rows can carry 1801 — so 1801
means a *technically clean action*, not a contest outcome. The proof: far more
than 50% of duels carry it, which is impossible for a win flag.

In [15]:
duels = [e for e in wc_events if e["eventName"] == "Duel"]
d1801 = sum(1 for e in duels if any(t["id"] == 1801 for t in e["tags"]))
print(f"World Cup duels: {len(duels):,}")
print(f"  carrying tag 1801: {d1801:,} ({100*d1801/len(duels):.1f}%)")
print("  > 50% => 1801 cannot mean 'won' (a contest has exactly one winner).")
print("  Conclusion: DEFENDING is built on VOLUME (clearances, duel counts,")
print("  recoveries), never on a duel win-rate, which this data cannot give us.")

World Cup duels: 25,927
  carrying tag 1801: 15,843 (61.1%)
  > 50% => 1801 cannot mean 'won' (a contest has exactly one winner).
  Conclusion: DEFENDING is built on VOLUME (clearances, duel counts,
  recoveries), never on a duel win-rate, which this data cannot give us.


## 8. Shot-on-target = tag 1801  (decision #2)

On a **Shot**, the clean-action tag 1801 *is* the on-target signal: every shot
carries exactly one of 1801 (on target / accurate) or 1802 (off target), and
every goal carries 1801. We confirm the partition holds with no overlap and no
gaps.

In [16]:
shots = [e for e in wc_events if e["eventName"] == "Shot"]
both  = sum(1 for e in shots if {1801, 1802} <= {t["id"] for t in e["tags"]})
neither = sum(1 for e in shots if not ({1801, 1802} & {t["id"] for t in e["tags"]}))
on_t  = sum(1 for e in shots if any(t["id"] == 1801 for t in e["tags"]))
goals_1801 = sum(1 for e in shots
                 if any(t["id"] == 101 for t in e["tags"])
                 and any(t["id"] == 1801 for t in e["tags"]))
goals_total = sum(1 for e in shots if any(t["id"] == 101 for t in e["tags"]))
print(f"shots: {len(shots):,}")
print(f"  carrying BOTH 1801 & 1802 : {both}  (must be 0)")
print(f"  carrying NEITHER          : {neither}  (must be 0)")
print(f"  on target (1801)          : {on_t:,}  ({100*on_t/len(shots):.0f}%)")
print(f"  goals also carrying 1801  : {goals_1801}/{goals_total}  (every goal is on target)")

shots: 1,419
  carrying BOTH 1801 & 1802 : 0  (must be 0)
  carrying NEITHER          : 0  (must be 0)
  on target (1801)          : 438  (31%)
  goals also carrying 1801  : 128/128  (every goal is on target)


## 9. Coordinates are team-relative  (decision #6)

Wyscout coordinates are 0–100 and **team-relative**: x=0 is a team's own goal
line, x=100 the opponent's. So *every* team attacks toward x=100 regardless of
which physical end they're on — which is what lets `end_x > start_x + 10` be a
universal "progressive pass" test. If this held, shots should originate near
x=100. They do (median ≈ 87, the attacking third). We confirm it on the World
Cup *and* on a league file (guarding against a per-competition convention
difference).

In [17]:
wc_sx = np.array([e["positions"][0]["x"] for e in shots if e.get("positions")])
print(f"World Cup shot start_x : median {np.median(wc_sx):.0f}  mean {wc_sx.mean():.1f}  "
      f"[{wc_sx.min()}–{wc_sx.max()}]")
# league cross-check from the cached enriched parquet (instant)
esp = pd.read_parquet(ENR / "spain.parquet", columns=["is_shot", "start_x"])
esp_sx = esp.loc[esp.is_shot, "start_x"].dropna()
print(f"La Liga   shot start_x : median {esp_sx.median():.0f}  mean {esp_sx.mean():.1f}  "
      f"[{esp_sx.min():.0f}–{esp_sx.max():.0f}]")
print("\nBoth cluster in the attacking third => team-relative convention holds for")
print("leagues and tournaments alike. Progressive-pass / final-third flags are valid.")

World Cup shot start_x : median 87  mean 84.3  [10–99]
La Liga   shot start_x : median 88  mean 85.2  [1–100]

Both cluster in the attacking third => team-relative convention holds for
leagues and tournaments alike. Progressive-pass / final-third flags are valid.


## 10. Minutes & extra time  (decision #7)

Substitutions are recorded with the minute they occurred. The max sub minute
in the World Cup exceeds 90 — proof some matches ran to extra time, so minutes
must be capped at 120 (not 90) for those games. We also confirm shootout
events exist and are correctly walled off from goal counts (§6).

In [18]:
sub_minutes = [wl._as_int(s.get("minute")) for mt in wc_matches
               for td in mt["teamsData"].values()
               for s in (td["formation"].get("substitutions") or [])
               if isinstance(s, dict)]
sub_minutes = [x for x in sub_minutes if x > 0]
over90 = sum(1 for x in sub_minutes if x > 90)
print(f"World Cup substitutions: {len(sub_minutes):,}")
print(f"  max sub minute        : {max(sub_minutes)}  (> 90 => extra time happened)")
print(f"  subs after minute 90  : {over90}")
et_matches = sum(1 for mt in wc_matches
                 if any(td.get("scoreET") not in (None, "null", 0, "0")
                        for td in mt["teamsData"].values()))
print(f"  matches with an ET score: {et_matches}")
print(f"  shootout (period P) events in WC: "
      f"{sum(1 for e in wc_events if e['matchPeriod']=='P'):,} "
      f"(scored via tag 101 but EXCLUDED from competitive goals)")

World Cup substitutions: 381
  max sub minute        : 119  (> 90 => extra time happened)
  subs after minute 90  : 31
  matches with an ET score: 5
  shootout (period P) events in WC: 76 (scored via tag 101 but EXCLUDED from competitive goals)


## Audit complete — locked decisions

Every rule below is now backed by a number recomputed above, not an assumption:

| Decision | Evidence (this notebook) |
|---|---|
| **Goals from the match sheet, not tag 101** | naïve tag-101 ≈ 2× real; lineup ↔ filtered-event agree ~100%; Kane = 6 (§6) |
| **Shot-on-target = tag 1801** | every shot has exactly one of 1801/1802; every goal has 1801 (§8) |
| **DEFENDING uses volume, not duel win-rate** | 61% of duels carry 1801 → not a "won" flag (§7) |
| **Assists from event tag 301** | lineup `assists` = 0 across all five leagues (§4B) |
| **Cards are minutes → 0/1 + minutes trim** | non-zero card values span 1–120 (§4A) |
| **Coordinates team-relative** | shots originate at x≈87 in both WC and La Liga (§9) |
| **Extra time → cap 120; shootout excluded** | max sub minute > 90; period P present (§10) |
| **Decode `\uXXXX` names; drop 237 coachless team-matches** | counts measured live (§2, §4C) |

With the data understood and every trap mapped, **Phase 1 (`01_master_tables`)**
can safely build the foundation tables.